# Recipe RAG: compute embeddings on a Colab GPU

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. When asked, allow access to Google Drive, then upload `colab_jobs.parquet`
   (or put it beforehand in `My Drive/recipe_rag/` to skip the upload).
3. Results are written to `My Drive/recipe_rag/embeddings/`. The run is resumable:
   if Colab disconnects, just run all cells again.
4. Download the `embeddings` folder from Google Drive and unzip its *content* into the
   project's `data/index/cache/` folder.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -U sentence-transformers

In [ ]:
import os, shutil
from google.colab import drive, files
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/recipe_rag'
OUT = f'{WORK}/embeddings'
os.makedirs(OUT, exist_ok=True)
JOBS = f'{WORK}/colab_jobs.parquet'
if not os.path.exists(JOBS):
    print('Upload colab_jobs.parquet ...')
    uploaded = files.upload()
    shutil.move(next(iter(uploaded)), JOBS)
print('jobs file:', JOBS, round(os.path.getsize(JOBS) / 1e6), 'MB')

In [ ]:
import hashlib, json, time
import numpy as np, pyarrow.parquet as pq, torch
from sentence_transformers import SentenceTransformer

SHARD = 10_000  # must match recipe_rag/build_index.py
table = pq.read_table(JOBS)
jobs = json.loads(table.schema.metadata[b'jobs'])
df = table.to_pandas()
sets = {name: g.sort_values('idx')['text'].tolist() for name, g in df.groupby('text_set')}
del df, table
for j in jobs:
    print(f"{j['out_dir']:55} {len(j['digests']):3d} shards")

In [ ]:
models = {}
for job in jobs:
    name = job['model']
    if name not in models:
        m = SentenceTransformer(name, device='cuda')
        m.half()  # fp16 on the GPU: ~4x faster, cosine vs fp32 > 0.999
        models[name] = m
    model = models[name]
    prefix = 'passage: ' if 'e5' in name.lower() else ''
    texts = sets[job['text_set']]
    out_dir = f"{OUT}/{job['out_dir']}"
    os.makedirs(out_dir, exist_ok=True)
    for i, expected in job['digests'].items():
        i = int(i)
        shard = texts[i * SHARD:(i + 1) * SHARD]
        digest = hashlib.md5(''.join(shard).encode()).hexdigest()[:12]
        assert digest == expected, f'text mismatch in {job["out_dir"]} shard {i}'
        path = f'{out_dir}/shard_{i:04d}_{digest}.npy'
        if os.path.exists(path):
            continue
        t0 = time.time()
        vecs = model.encode([prefix + t for t in shard], batch_size=256,
                            normalize_embeddings=True, convert_to_numpy=True)
        np.save(path, vecs.astype('float16'))
        print(f"{job['out_dir']} shard {i}: {len(shard) / (time.time() - t0):.0f} texts/s", flush=True)
print('All done.')

In [ ]:
# Optional: zip the results to download a single file instead of the folder
!cd "$OUT" && zip -qr /content/drive/MyDrive/recipe_rag/embeddings.zip . && ls -lh /content/drive/MyDrive/recipe_rag/